## Ingestion of cleaned BI ready tables into gold layer ##

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql import Row
catalog_name = "ecommerce"

## dim_products table joined with brand and category table ##

In [0]:
df_silver_products = spark.read.table(f"{catalog_name}.silver.slv_products")
df_silver_brands = spark.read.table(f"{catalog_name}.silver.slv_brands")
df_silver_category = spark.read.table(f"{catalog_name}.silver.slv_category")
df_silver_products.createOrReplaceTempView("v_products")
df_silver_brands.createOrReplaceTempView("v_brands")
df_silver_category.createOrReplaceTempView("v_category")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql("SELECT * FROM v_products").show(10)
spark.sql("SELECT * FROM v_brands").show(10)
spark.sql("SELECT * FROM v_category").show(10)

## building brands and category mapping table and writing into gold layer

spark.sql(f"""

CREATE OR REPLACE TABLE ecommerce.gold.gld_dim_products AS
WITH brand_category AS(
    SELECT 
        b.brand_code,
        b.brand_name,
        c.category_code,
        c.category_name
    FROM v_brands b
    JOIN v_category c
    ON b.category_code = c.category_code
)
SELECT
    p.product_id,
    p.sku,
    p.category_code,
    COALESCE(bc.category_name, "Not Available") AS category_name,
    p.brand_code,
    COALESCE(bc.brand_name,"Not Available") AS brand_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    p.rating_count,
    p._source_file,
    p._ingested_at
    
    
FROM v_products p
LEFT JOIN brand_category bc
ON p.brand_code = bc.brand_code
JOIN v_category c
ON p.category_code = bc.category_code and p.brand_code = bc.brand_code
"""
)
display(spark.sql("SELECT * FROM ecommerce.gold.gld_dim_products").limit(10))



## Gold layer customers table ##

In [0]:

# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  

# 1 Flatten country_state_map into a list of Rows
rows = []
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))
rows[:10] 

# 2️ Create mapping DataFrame
df_region_mapping = spark.createDataFrame(rows)

# Optional: show mapping
df_region_mapping.show(truncate=False)
df_silver_customers = spark.read.table(f"{catalog_name}.silver.slv_customers")
df_silver_customers.show(10)
df_gold_customers = df_silver_customers.join(df_region_mapping, on = ["country","state"] , how ="left")
df_gold_customers.fillna("other", subset=["region"])

customers_column_order = ["customer_id","phone","country_code","country","state","region","_source_file","_ingested_at"]
df_gold_customers = df_gold_customers.select(*customers_column_order)
display(df_gold_customers.limit(10))
### writing into gold layer
df_gold_customers.write.mode("overwrite").format("delta").option("mergeSchema","true").saveAsTable(f"{catalog_name}.gold.gld_dim_customers")



### date table ###

In [0]:
df_silver_date = spark.read.table(f"{catalog_name}.silver.slv_date")
df_silver_date.show(10)
df_gold_date = df_silver_date.withColumn("dateid",F.date_format(F.col("date"),"yyyyMMdd")).withColumn("month",F.date_format(F.col("date"),"MMMM"))

df_gold_date = df_gold_date.withColumn("is_weekend",F.when(F.col("day_name").isin(["Saturday","Sunday"]),1).otherwise(0))
display(df_gold_date.limit(10))
date_column_order = ["dateid","date","year","month","day_name","quarter","week_of_year","is_weekend","_source_file","_ingested_at"]
df_gold_date = df_gold_date.select(*date_column_order)
display(df_gold_date.limit(10))
### writing into gold layer
df_gold_date.write.mode("overwrite").format("delta").option("mergeSchema","true").saveAsTable(f"{catalog_name}.gold.gld_dim_date")